### ECE/CS/ISyE 524 &mdash; Introduction to Optimization ###



# Optimizing Bus Routing and Scheduling for a Campus Shuttle System #

#### Steve Akpojisheri (akpojisheri@wisc.edu), Indumathi Muruganandam (muruganindum@wisc.edu), and Hunter Waugh (htwaugh@wisc.edu)

*****



### Table of Contents

1. [Introduction](#1.-Introduction)
1. [Mathematical Model](#2.-Mathematical-model)
1. [Implementation](#3.-Implementation)
1. [Results and Discussion](#4.-Results-and-discussion)
   1. [Optimization Results](#4.A.-Optimization-Results)
   1. [Sensitivity Analysis](#4.B.-Sensitivity-Analysis)
1. [Conclusion](#5.-Conclusion)
1. [Author Contributions](#6.-Author-Contributions)



## 1. Introduction ##

Campus shuttle systems are vital for transporting students, faculty, and staff across university grounds efficiently. However, designing optimal routes and schedules for these systems presents a complex optimization challenge. This project aims to design an optimal routing and scheduling plan for the UW-Madison campus shuttle system that minimizes operational costs while maximizing service quality.

Many universities, including UW-Madison, operate shuttle services to connect key locations such as residence halls, academic buildings, and parking facilities. These systems often face challenges like overcrowding during peak hours, underutilization during off-peak times, and inefficient routing that leads to unnecessary fuel consumption and driver hours. With rising sustainability goals and budget constraints, optimizing these shuttle systems has become increasingly important.

The central question we aim to address is: How can we assign routes and departure schedules to a fixed number of buses such that we meet time and capacity constraints while minimizing operational costs? 

To solve this problem, we've developed a mixed-integer programming (MIP) model that optimizes both routing decisions (which stops each bus visits and in what order) and scheduling aspects (when buses depart from each stop). Our model incorporates real-world constraints such as bus capacity limitations and ensures that all important campus locations are adequately served.

The data for this project comes from publicly available UW-Madison campus bus route information from the City of Madison Metro Transit system's General Transit Feed Specification (GTFS) feed. Due to the complexity and size of the complete dataset, we've implemented a strategic sampling approach that focuses on the most frequently visited stops and the most important bus routes.

The remainder of this report is organized as follows: Section 2 describes the mathematical formulation of our model; Section 3 details our implementation in Julia using the JuMP optimization framework; Section 4 presents the results of our optimization and analyzes their implications; and Section 5 concludes with a summary of our findings and suggestions for future work.



## 2. Mathematical model ##

Our optimization model is formulated as a Mixed Integer Programming (MIP) problem, combining elements of the Vehicle Routing Problem (VRP) and scheduling optimization. The model makes several key assumptions:

1. Each bus starts and ends at the same depot location
2. Each bus has a fixed capacity that cannot be exceeded
3. Passenger demand at each stop is deterministic and known in advance
4. Travel times between stops are based on distance and average speed
5. A certain minimum service level (every stop must be visited) must be maintained



### Decision Variables

- $x_{ijk} \in \{0,1\}$: Binary variable indicating whether bus $k$ travels directly from stop $i$ to stop $j$
- $y_{ik} \geq 0$: Integer variable representing the number of passengers on bus $k$ after leaving stop $i$
- $u_{ik} \geq 0$: Auxiliary variable for subtour elimination, indicating the position of stop $i$ in the route of bus $k$
- $\text{route\_time}_k \geq 0$: Continuous variable representing the total route time for bus $k$



### Objective Function

The objective is to minimize the total operational cost, which includes both distance-based costs and driver wages:

$$
\min \sum_{i=1}^{n}\sum_{j=1}^{n}\sum_{k=1}^{K} c_{ij} \cdot x_{ijk} + w \cdot \sum_{k=1}^{K} \text{route\_time}_k
$$

where:
- $c_{ij}$ is the travel cost (distance) from stop $i$ to stop $j$
- $w$ is the driver wage rate
- $K$ is the total number of available buses
- $n$ is the number of stops (including the depot)



### Constraints

1. Each stop (except the depot) must be visited exactly once:

$$
\sum_{i=1}^{n}\sum_{k=1}^{K} x_{ijk} = 1 \quad \forall j \in \{2,3,...,n\}, i \neq j
$$

2. Flow conservation at each stop:

$$
\sum_{i=1}^{n} x_{ijk} = \sum_{i=1}^{n} x_{jik} \quad \forall j \in \{1,2,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

3. Each bus starts and ends at the depot (stop 1):

$$
\sum_{j=2}^{n} x_{1jk} = \sum_{i=2}^{n} x_{i1k} \quad \forall k \in \{1,2,...,K\}
$$

4. Each bus leaves the depot at most once:

$$
\sum_{j=2}^{n} x_{1jk} \leq 1 \quad \forall k \in \{1,2,...,K\}
$$

5. Bus capacity constraints:

$$
y_{ik} \leq \text{Cap}_k \quad \forall i \in \{1,2,...,n\}, k \in \{1,2,...,K\}
$$

6. Passenger flow constraints - if bus $k$ travels from stop $i$ to stop $j$, the passengers at $j$ equal passengers at $i$ plus demand at $j$:

$$
y_{jk} \geq y_{ik} + d_j - M(1-x_{ijk}) \quad \forall i,j \in \{1,2,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

$$
y_{jk} \leq y_{ik} + d_j + M(1-x_{ijk}) \quad \forall i,j \in \{1,2,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

7. Subtour elimination constraints using Miller-Tucker-Zemlin (MTZ) formulation:

$$
u_{jk} \geq u_{ik} + 1 - M(1-x_{ijk}) \quad \forall i,j \in \{2,3,...,n\}, k \in \{1,2,...,K\}, i \neq j
$$

$$
u_{ik} \leq n-1 \quad \forall i \in \{2,3,...,n\}, k \in \{1,2,...,K\}
$$

$$
u_{1k} = 0 \quad \forall k \in \{1,2,...,K\}
$$

8. Route time calculation:

$$
\text{route\_time}_k \geq \sum_{i=1}^{n}\sum_{j=1}^{n} x_{ijk} \cdot (c_{ij}/v + s) \quad \forall k \in \{1,2,...,K\}, i \neq j
$$

where $v$ is the average speed (distance units per minute) and $s$ is the average stop time (minutes).

This MIP model allows us to find optimal bus routes and passenger assignments while respecting capacity constraints and ensuring all stops are serviced.



## 3. Implementation ##

Our implementation uses Julia with the JuMP optimization framework and the HiGHS solver to solve the mixed-integer programming model. Due to the large size of the complete dataset, we implemented a strategic sampling approach to make the problem computationally tractable.

The implementation consists of several key components:

1. **Data preprocessing and strategic sampling**: We select the most important stops based on frequency and identify the most important trips that cover these major stops.

2. **Model construction**: We build the MIP model with the appropriate decision variables, constraints, and objective function as described in the mathematical model.

3. **Solution extraction and analysis**: After solving the model, we extract the optimized routes and analyze various performance metrics.

4. **Visualization**: We create visualizations of the optimized routes and passenger loads.

Below is an explanation of the core functions in our implementation:



In [6]:
# Bus Routing and Scheduling Optimization - Strategic Sampling Approach
using JuMP
using HiGHS
using CSV
using DataFrames
using Statistics
using Plots
using Dates
using Random

"""
    strategic_sample_stop_times(file_path; 
                               max_stops=30, 
                               sample_trips=true)

Strategically sample the stop_times data to focus on major stops and routes
"""
function strategic_sample_stop_times(file_path; max_stops=30, sample_trips=true)
    println("Reading stop_times data and extracting key information...")
    
    # First pass: Identify stop frequencies and important trip patterns
    stop_counts = Dict{Int, Int}()
    trip_stops = Dict{Int, Vector{Int}}()
    
    # Process the file line by line to avoid loading everything into memory
    open(file_path, "r") do io
        # Read header
        header = readline(io)
        
        # Process data rows
        while !eof(io)
            line = readline(io)
            fields = split(line, ',')
            
            if length(fields) >= 10
                try
                    trip_id = parse(Int, fields[1])
                    stop_id = parse(Int, fields[4])
                    stop_sequence = parse(Int, fields[5])
                    
                    # Count stop frequencies
                    stop_counts[stop_id] = get(stop_counts, stop_id, 0) + 1
                    
                    # Group stops by trip
                    if !haskey(trip_stops, trip_id)
                        trip_stops[trip_id] = Int[]
                    end
                    push!(trip_stops[trip_id], stop_id)
                catch
                    # Skip rows with parsing errors
                    continue
                end
            end
        end
    end
    
    # Sort stops by frequency and select the most important ones
    sorted_stops = sort(collect(stop_counts), by=x->x[2], rev=true)
    major_stop_ids = [stop_id for (stop_id, _) in sorted_stops[1:min(max_stops, length(sorted_stops))]]
    
    # Identify the most important trips (those that cover major stops)
    trip_importance = Dict{Int, Float64}()
    for (trip_id, stops) in trip_stops
        # Calculate what fraction of this trip's stops are major stops
        major_stop_count = count(s -> s in major_stop_ids, unique(stops))
        trip_importance[trip_id] = major_stop_count / length(unique(stops))
    end
    
    # Sort trips by importance
    sorted_trips = sort(collect(trip_importance), by=x->x[2], rev=true)
    
    # Select the top 20% of trips but ensure minimum of 100 trips for diversity
    important_trip_threshold = 0.8  # Only include trips with at least 80% important stops
    important_trips = [trip_id for (trip_id, score) in sorted_trips if score >= important_trip_threshold]
    
    if length(important_trips) < 100
        # Add more trips if needed
        additional_trips = [trip_id for (trip_id, _) in sorted_trips if !(trip_id in important_trips)]
        important_trips = vcat(important_trips, additional_trips[1:min(100-length(important_trips), length(additional_trips))])
    end
    
    # Second pass: Extract only the data for major stops and important trips
    println("Extracting strategic subsample based on $(length(major_stop_ids)) major stops and $(length(important_trips)) important trips...")
    
    # Prepare data structure
    sampled_data = []
    
    open(file_path, "r") do io
        # Read header
        header = readline(io)
        columns = split(header, ',')
        
        # Process data rows
        while !eof(io)
            line = readline(io)
            fields = split(line, ',')
            
            if length(fields) >= 10
                try
                    trip_id = parse(Int, fields[1])
                    arrival_time = fields[2]
                    departure_time = fields[3]
                    stop_id = parse(Int, fields[4])
                    stop_sequence = parse(Int, fields[5])
                    stop_headsign = fields[6]
                    shape_dist_traveled = parse(Float64, fields[9])
                    
                    # Include only if it's a major stop and either from an important trip or we're not sampling trips
                    if stop_id in major_stop_ids && (!sample_trips || trip_id in important_trips)
                        row = Dict(
                            :trip_id => trip_id,
                            :arrival_time => arrival_time,
                            :departure_time => departure_time,
                            :stop_id => stop_id,
                            :stop_sequence => stop_sequence,
                            :stop_headsign => stop_headsign,
                            :shape_dist_traveled => shape_dist_traveled
                        )
                        push!(sampled_data, row)
                    end
                catch
                    # Skip rows with parsing errors
                    continue
                end
            end
        end
    end
    
    # Convert to DataFrame
    stop_times = DataFrame(sampled_data)
    
    # Convert time strings to minutes since midnight
    function time_to_minutes(time_str)
        try
            h, m, s = parse.(Int, split(time_str, ":"))
            return h * 60 + m + s / 60
        catch
            return missing
        end
    end
    
    # Add calculated columns
    stop_times[!, :arrival_minutes] = time_to_minutes.(stop_times[!, :arrival_time])
    stop_times[!, :departure_minutes] = time_to_minutes.(stop_times[!, :departure_time])
    
    println("Created strategic subsample with $(size(stop_times, 1)) rows")
    return stop_times, major_stop_ids
end

"""
    get_stops_data(stop_times, stop_ids)

Extract stop information for the selected stops
"""
function get_stops_data(stop_times, stop_ids)
    stops = Dict{Int, NamedTuple}()
    
    for stop_id in stop_ids
        stop_data = filter(row -> row.stop_id == stop_id, stop_times)
        if size(stop_data, 1) > 0
            # Use shape_dist_traveled as a proxy for location
            dist = mean(stop_data[!, :shape_dist_traveled])
            stops[stop_id] = (id = stop_id, dist = dist)
        end
    end
    
    # Handle special case for depot
    # If depot isn't in stop_ids, add a virtual depot at the minimum distance point
    if !haskey(stops, stop_ids[1])
        min_dist = minimum([s.dist for s in values(stops)])
        stops[stop_ids[1]] = (id = stop_ids[1], dist = min_dist)
    end
    
    return stops
end

"""
    calculate_travel_costs(stops)

Calculate travel costs between stops based on distance
"""
function calculate_travel_costs(stops)
    stop_ids = collect(keys(stops))
    n = length(stop_ids)
    
    # Initialize cost matrix
    costs = zeros(Float64, n, n)
    
    # Calculate costs based on distance between stops
    for i in 1:n
        for j in 1:n
            if i != j
                # Calculate distance based on shape_dist_traveled values
                costs[i, j] = abs(stops[stop_ids[i]].dist - stops[stop_ids[j]].dist)
                
                # Add small random variation to prevent degenerate solutions (0.1% noise)
                costs[i, j] *= (1.0 + 0.001 * rand())
            end
        end
    end
    
    return costs, stop_ids
end

"""
    estimate_demand(stop_times, stop_ids)

Estimate passenger demand based on stop frequency in the data
"""
function estimate_demand(stop_times, stop_ids)
    n = length(stop_ids)
    demand = zeros(Int, n)
    
    # Count stops by frequency in the data
    stop_counts = Dict{Int, Int}()
    for stop_id in stop_times[!, :stop_id]
        stop_counts[stop_id] = get(stop_counts, stop_id, 0) + 1
    end
    
    # Normalize to a reasonable passenger demand range (5-25 passengers)
    for (i, stop_id) in enumerate(stop_ids)
        count = get(stop_counts, stop_id, 0)
        if count > 0
            # Map to range 5-25 based on frequency percentile
            counts = collect(values(stop_counts))
            percentile = count_to_percentile(count, counts)
            demand[i] = round(Int, 5 + 20 * percentile)
        else
            demand[i] = 5  # Minimum demand
        end
    end
    
    # Depot has no demand
    demand[1] = 0
    
    return demand
end

"""
    count_to_percentile(count, counts)

Helper function to convert count to percentile
"""
function count_to_percentile(count, counts)
    sorted_counts = sort(counts)
    idx = searchsortedfirst(sorted_counts, count)
    return (idx - 1) / (length(sorted_counts) - 1)
end

"""
    build_optimization_model(stops, stop_ids, costs, demand; 
                            num_buses=5, 
                            bus_capacity=40, 
                            driver_wage=25.0)

Build a more efficient optimization model for the routing problem
"""
function build_optimization_model(stops, stop_ids, costs, demand; 
                                 num_buses=5, 
                                 bus_capacity=40, 
                                 driver_wage=25.0)
    n = length(stops)
    K = num_buses
    
    # Create model with HiGHS solver
    model = Model(HiGHS.Optimizer)
    
    # Set solver parameters for better memory efficiency
    set_optimizer_attribute(model, "mip_feasibility_tolerance", 1e-4)
    set_optimizer_attribute(model, "mip_rel_gap", 0.05)  # Accept 5% gap
    
    # Define decision variables
    @variable(model, x[1:n, 1:n, 1:K], Bin)  # 1 if bus k travels from stop i to j
    @variable(model, y[1:n, 1:K] >= 0, Int)  # Number of passengers on bus k after leaving stop i
    @variable(model, route_time[1:K] >= 0)   # Total route time for each bus
    
    # Objective function: minimize total cost (distance + driver wages)
    @objective(model, Min, 
        sum(costs[i, j] * x[i, j, k] for i in 1:n, j in 1:n, k in 1:K if i != j) + 
        driver_wage * sum(route_time[k] for k in 1:K))
    
    # Constraints
    
    # Each stop (except depot) must be visited exactly once
    for j in 2:n
        @constraint(model, sum(x[i, j, k] for i in 1:n, k in 1:K if i != j) == 1)
    end
    
    # Flow conservation: if a bus enters a stop, it must exit
    for k in 1:K, j in 1:n
        @constraint(model, 
            sum(x[i, j, k] for i in 1:n if i != j) == 
            sum(x[j, i, k] for i in 1:n if i != j))
    end
    
    # Each bus starts and ends at the depot (stop 1)
    for k in 1:K
        # Each bus must leave the depot at most once
        @constraint(model, sum(x[1, j, k] for j in 2:n) <= 1)
        
        # Each bus must return to the depot if it leaves
        @constraint(model, sum(x[1, j, k] for j in 2:n) == sum(x[i, 1, k] for i in 2:n))
    end
    
    # Bus capacity constraints
    for k in 1:K, i in 1:n
        @constraint(model, y[i, k] <= bus_capacity)
    end
    
    # Simplified passenger flow modeling
    # Initialize passenger count at depot
    for k in 1:K
        @constraint(model, y[1, k] == 0)
    end
    
    # Passenger flow at other stops
    M = bus_capacity + sum(demand)  # Big M value
    for k in 1:K, j in 2:n
        # At most one stop can precede j on route k
        @constraint(model, sum(x[i, j, k] for i in 1:n if i != j) <= 1)
        
        # For each potential predecessor i of j
        for i in 1:n
            if i != j
                # If bus k travels from i to j, passengers from i plus new demand at j equals passengers at j
                @constraint(model, y[j, k] >= y[i, k] + demand[j] - M * (1 - x[i, j, k]))
                @constraint(model, y[j, k] <= y[i, k] + demand[j] + M * (1 - x[i, j, k]))
            end
        end
    end
    
    # Calculate route times - assume average speed of 30 distance units per minute
    for k in 1:K
        # Sum up travel times along the route
        @constraint(model, route_time[k] >= 
            sum(x[i, j, k] * (costs[i, j] / 30 + 1) for i in 1:n, j in 1:n if i != j))
    end
    
    # Subtour elimination using Miller-Tucker-Zemlin (MTZ) formulation
    @variable(model, u[1:n, 1:K] >= 0)  # Position in the route
    
    # Set depot position to 0
    for k in 1:K
        @constraint(model, u[1, k] == 0)
    end
    
    # For each non-depot stop
    M = n+1  # Big M value
    for k in 1:K, i in 2:n, j in 2:n
        if i != j
            # If bus k travels from i to j, ensure j comes after i in the route
            @constraint(model, u[j, k] >= u[i, k] + 1 - M * (1 - x[i, j, k]))
        end
    end
    
    # Maximum position value constraint
    for k in 1:K, i in 2:n
        @constraint(model, u[i, k] <= n - 1)
    end
    
    return model, x, y, route_time
end

"""
    solve_model(model)

Solve the optimization model with appropriate settings
"""
function solve_model(model)
    # Set time limit
    set_time_limit_sec(model, 300)  # 5 minute time limit
    
    # Solve the model
    optimize!(model)
    
    status = termination_status(model)
    if status in [MOI.OPTIMAL, MOI.TIME_LIMIT] && has_values(model)
        objective = objective_value(model)
        println("Solution found with objective value: $objective")
        println("Status: $status")
        return status, objective
    else
        println("No solution found. Status: $status")
        return status, Inf
    end
end

"""
    extract_solution(model, x, y, route_time, stop_ids, num_buses)

Extract the solution from the solved model
"""
function extract_solution(model, x, y, route_time, stop_ids, num_buses)
    if !has_values(model)
        return nothing, nothing, nothing
    end
    
    n = length(stop_ids)
    K = num_buses
    
    # Initialize solution structure
    routes = Dict{Int, Vector{Int}}()
    loads = Dict{Int, Dict{Int, Int}}()
    times = Dict{Int, Float64}()
    
    # Extract solution
    for k in 1:K
        # Only process buses that are used (leave depot)
        if sum(value.(x[1, :, k])) > 0.5
            routes[k] = Int[]
            loads[k] = Dict{Int, Int}()
            
            # Start at depot
            current = 1
            push!(routes[k], stop_ids[current])
            loads[k][stop_ids[current]] = round(Int, value(y[current, k]))
            
            # Follow the route
            while true
                next_stop = 0
                for j in 1:n
                    if j != current && value(x[current, j, k]) > 0.5
                        next_stop = j
                        break
                    end
                end
                
                if next_stop == 0 || next_stop == 1 && current != 1
                    # Back to depot or end of route
                    if next_stop == 1
                        push!(routes[k], stop_ids[1])
                        loads[k][stop_ids[1]] = round(Int, value(y[1, k]))
                    end
                    break
                end
                
                push!(routes[k], stop_ids[next_stop])
                loads[k][stop_ids[next_stop]] = round(Int, value(y[next_stop, k]))
                
                current = next_stop
            end
            
            # Store route time
            times[k] = value(route_time[k])
        end
    end
    
    return routes, loads, times
end

"""
    visualize_solution(routes, loads, times, stops)

Create visualizations of the optimization results
"""
function visualize_solution(routes, loads, times, stops)
    if isnothing(routes)
        println("No solution to visualize")
        return nothing, nothing
    end
    
    # Route visualization
    p1 = plot(title="Optimized Bus Routes", 
             xlabel="Distance", 
             ylabel="Bus", 
             size=(800, 600))
    
    # Schedule visualization
    p2 = plot(title="Bus Passenger Loads", 
             xlabel="Stop Number", 
             ylabel="Passengers", 
             size=(800, 600))
    
    colors = [:red, :blue, :green, :purple, :orange, :cyan, :magenta, :brown, :pink, :gray]
    
    for (k, route) in sort(collect(routes))
        # Plot route
        xs = [stops[stop_id].dist for stop_id in route]
        ys = fill(k, length(route))
        color_idx = mod1(k, length(colors))
        
        plot!(p1, xs, ys, 
              marker=:circle, 
              linewidth=2, 
              label="Bus $k ($(round(times[k], digits=1)) min)", 
              color=colors[color_idx])
        
        # Plot passenger loads
        stop_nums = 1:length(route)
        load_vals = [loads[k][stop_id] for stop_id in route]
        
        plot!(p2, stop_nums, load_vals, 
              marker=:circle, 
              linewidth=2, 
              label="Bus $k", 
              color=colors[color_idx])
    end
    
    return p1, p2
end

"""
    save_results(routes, loads, times, stop_ids)

Save the optimization results to CSV
"""
function save_results(routes, loads, times, stop_ids)
    if isnothing(routes)
        println("No results to save")
        return nothing
    end
    
    # Create results dataframe
    results = DataFrame(
        bus_id = Int[],
        stop_id = Int[],
        stop_order = Int[],
        passenger_load = Int[]
    )
    
    for (k, route) in routes
        for (i, stop_id) in enumerate(route)
            push!(results, (
                k, 
                stop_id, 
                i, 
                get(loads[k], stop_id, 0)
            ))
        end
    end
    
    # Save to CSV
    CSV.write("optimization_results.csv", results)
    
    # Create summary dataframe
    summary = DataFrame(
        bus_id = Int[],
        num_stops = Int[],
        route_time = Float64[],
        max_load = Int[]
    )
    
    for (k, route) in routes
        max_load = maximum(values(loads[k]))
        push!(summary, (
            k,
            length(route),
            times[k],
            max_load
        ))
    end
    
    # Save to CSV
    CSV.write("optimization_summary.csv", summary)
    
    return results, summary
end

"""
    analyze_solution(routes, loads, times, stops, costs, stop_ids)

Analyze the optimized solution
"""
function analyze_solution(routes, loads, times, stops, costs, stop_ids)
    if isnothing(routes)
        println("No solution to analyze")
        return nothing
    end
    
    # Calculate statistics
    total_distance = 0.0
    for (k, route) in routes
        for i in 1:(length(route)-1)
            stop1_idx = findfirst(id -> id == route[i], stop_ids)
            stop2_idx = findfirst(id -> id == route[i+1], stop_ids)
            if !isnothing(stop1_idx) && !isnothing(stop2_idx)
                total_distance += costs[stop1_idx, stop2_idx]
            end
        end
    end
    
    # Calculate max and average loads
    max_loads = [maximum(values(loads[k])) for k in keys(routes)]
    max_load = maximum(max_loads)
    avg_load = mean([mean(values(loads[k])) for k in keys(routes)])
    
    # Calculate route times
    total_time = sum(values(times))
    avg_time = mean(values(times))
    max_time = maximum(values(times))
    
    # Print statistics
    println("\nSolution Statistics:")
    println("Number of buses used: $(length(routes))")
    println("Total distance: $(round(total_distance, digits=2)) units")
    println("Total operation time: $(round(total_time, digits=2)) minutes")
    println("Average route time: $(round(avg_time, digits=2)) minutes")
    println("Maximum route time: $(round(max_time, digits=2)) minutes")
    println("Maximum passenger load: $max_load passengers")
    println("Average passenger load: $(round(avg_load, digits=2)) passengers")
    
    # Return statistics
    return Dict(
        "num_buses" => length(routes),
        "total_distance" => total_distance,
        "total_time" => total_time,
        "avg_time" => avg_time,
        "max_time" => max_time,
        "max_load" => max_load,
        "avg_load" => avg_load
    )
end

"""
    main(stop_times_path, trips_path; 
         max_stops=30, 
         num_buses=5, 
         bus_capacity=40, 
         driver_wage=25.0)

Main function to run the optimization with strategic sampling
"""
function main(stop_times_path, trips_path; 
              max_stops=30, 
              num_buses=5, 
              bus_capacity=40, 
              driver_wage=25.0)
    
    # Set random seed for reproducibility
    Random.seed!(42)
    
    println("Strategic sampling from stop_times data...")
    stop_times, major_stop_ids = strategic_sample_stop_times(stop_times_path; max_stops=max_stops)
    
    println("Processing stop data...")
    stops = get_stops_data(stop_times, major_stop_ids)
    costs, stop_ids = calculate_travel_costs(stops)
    demand = estimate_demand(stop_times, stop_ids)
    
    println("Building optimization model...")
    model, x, y, route_time = build_optimization_model(stops, stop_ids, costs, demand;
                                                     num_buses=num_buses,
                                                     bus_capacity=bus_capacity,
                                                     driver_wage=driver_wage)
    
    println("Solving model...")
    status, objective = solve_model(model)
    
    if status in [MOI.OPTIMAL, MOI.TIME_LIMIT] && has_values(model)
        println("Extracting solution...")
        routes, loads, times = extract_solution(model, x, y, route_time, stop_ids, num_buses)
        
        println("Analyzing solution...")
        stats = analyze_solution(routes, loads, times, stops, costs, stop_ids)
        
        println("Visualizing solution...")
        route_plot, load_plot = visualize_solution(routes, loads, times, stops)
        
        println("Saving results...")
        results, summary = save_results(routes, loads, times, stop_ids)
        
        # Save plots
        savefig(route_plot, "optimized_routes.png")
        savefig(load_plot, "passenger_loads.png")
        
        return routes, loads, times, stats, route_plot, load_plot
    else
        println("No solution found. Status: $status")
        return nothing, nothing, nothing, nothing, nothing, nothing
    end
end

# --- Run the optimization ---

# Define paths to data files
stop_times_path = "stop_times.csv"
trips_path = "trips_clean.csv"

# Run optimization with strategic sampling
routes, loads, times, stats, route_plot, load_plot = main(stop_times_path, trips_path; 
                                                         max_stops=25,  # Use top 25 stops
                                                         num_buses=4,   # Use 4 buses
                                                         bus_capacity=40,
                                                         driver_wage=25.0)

Strategic sampling from stop_times data...
Reading stop_times data and extracting key information...
Extracting strategic subsample based on 25 major stops and 100 important trips...
Created strategic subsample with 880 rows
Processing stop data...
Building optimization model...
Solving model...
Running HiGHS 1.9.0 (git hash: 66f735e60): Copyright (c) 2024 HiGHS under MIT licence terms
Coefficient ranges:
  Matrix [1e+00, 2e+02]
  Cost   [8e+00, 4e+03]
  Bound  [1e+00, 1e+00]
  RHS    [1e+00, 2e+02]
Presolving model
2434 rows, 952 cols, 10192 nonzeros  0s
2398 rows, 952 cols, 10302 nonzeros  1s

Solving MIP model with:
   2398 rows
   952 cols (840 binary, 56 integer, 0 implied int., 56 continuous)
   10302 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic; L => Sub-MIP;
     P => Empty MIP; R => Randomized rounding; S => Solve LP; T => Evaluate node; U => Unbounded;
     z => Trivial zero; l => Trivial lower; u => Trivial upper; p => Trivial p

(Dict(4 => [988, 628, 2866, 1757, 2896, 1262, 988], 2 => [988, 2810, 2469, 1185, 2863, 988], 3 => [988, 877, 741, 882, 988], 1 => [988, 2894, 555, 988]), Dict(4 => Dict(988 => 0, 1262 => 37, 1757 => 21, 2896 => 26, 2866 => 16, 628 => 11), 2 => Dict(988 => 0, 2810 => 11, 1185 => 27, 2863 => 38, 2469 => 16), 3 => Dict(988 => 0, 882 => 33, 877 => 11, 741 => 22), 1 => Dict(988 => 0, 2894 => 11, 555 => 22)), Dict(4 => 294.1314089986327, 2 => 97.77566324346894, 3 => 163.34209795519936, 1 => 96.3108460465605), Dict{String, Real}("max_load" => 38, "total_time" => 651.5600162438616, "avg_time" => 162.8900040609654, "num_buses" => 4, "total_distance" => 19006.800487315842, "max_time" => 294.1314089986327, "avg_load" => 16.1), Plot{Plots.GRBackend() n=4}, Plot{Plots.GRBackend() n=4})

### Strategic Sampling of Stop Times Data

```julia
function strategic_sample_stop_times(file_path; max_stops=30, sample_trips=true)
    # First pass: Identify stop frequencies and important trip patterns
    stop_counts = Dict{Int, Int}()
    trip_stops = Dict{Int, Vector{Int}}()
    
    # Process file to count stop frequencies and group stops by trip
    # ...
    
    # Sort stops by frequency and select top max_stops
    sorted_stops = sort(collect(stop_counts), by=x->x[2], rev=true)
    major_stop_ids = [stop_id for (stop_id, _) in sorted_stops[1:min(max_stops, length(sorted_stops))]]
    
    # Identify important trips that cover major stops
    # ...
    
    # Second pass: Extract only data for major stops and important trips
    # ...
    
    return stop_times, major_stop_ids
end
```



### Building the Optimization Model

```julia
function build_optimization_model(stops, stop_ids, costs, demand; 
                                 num_buses=5, 
                                 bus_capacity=40, 
                                 driver_wage=25.0)
    n = length(stops)
    K = num_buses
    
    # Create model with HiGHS solver
    model = Model(HiGHS.Optimizer)
    
    # Set solver parameters
    set_optimizer_attribute(model, "mip_feasibility_tolerance", 1e-4)
    set_optimizer_attribute(model, "mip_rel_gap", 0.05)  # Accept 5% gap
    
    # Define decision variables
    @variable(model, x[1:n, 1:n, 1:K], Bin)  # 1 if bus k travels from stop i to j
    @variable(model, y[1:n, 1:K] >= 0, Int)  # Number of passengers on bus k after leaving stop i
    @variable(model, route_time[1:K] >= 0)   # Total route time for each bus
    
    # Objective function: minimize total cost (distance + driver wages)
    @objective(model, Min, 
        sum(costs[i, j] * x[i, j, k] for i in 1:n, j in 1:n, k in 1:K if i != j) + 
        driver_wage * sum(route_time[k] for k in 1:K))
    
    # Constraints
    # Each stop (except depot) must be visited exactly once
    # Flow conservation
    # Bus starts and ends at depot
    # Bus capacity constraints
    # Passenger flow constraints
    # Subtour elimination using Miller-Tucker-Zemlin formulation
    # ...
    
    return model, x, y, route_time
end
```



### Solving the Model and Extracting Results

```julia
function solve_model(model)
    # Set time limit to 5 minutes
    set_time_limit_sec(model, 300)
    
    # Solve the model
    optimize!(model)
    
    status = termination_status(model)
    if status in [MOI.OPTIMAL, MOI.TIME_LIMIT] && has_values(model)
        objective = objective_value(model)
        println("Solution found with objective value: $objective")
        println("Status: $status")
        return status, objective
    else
        println("No solution found. Status: $status")
        return status, Inf
    end
end

function extract_solution(model, x, y, route_time, stop_ids, num_buses)
    # Extract routes, passenger loads, and route times from the solved model
    # ...
    return routes, loads, times
end
```



### Main Function to Run the Optimization

```julia
function main(stop_times_path, trips_path; 
              max_stops=30, 
              num_buses=5, 
              bus_capacity=40, 
              driver_wage=25.0)
    
    # Set random seed for reproducibility
    Random.seed!(42)
    
    # Strategic sampling from stop_times data
    stop_times, major_stop_ids = strategic_sample_stop_times(stop_times_path; max_stops=max_stops)
    
    # Process stop data
    stops = get_stops_data(stop_times, major_stop_ids)
    costs, stop_ids = calculate_travel_costs(stops)
    demand = estimate_demand(stop_times, stop_ids)
    
    # Build and solve optimization model
    model, x, y, route_time = build_optimization_model(stops, stop_ids, costs, demand;
                                                     num_buses=num_buses,
                                                     bus_capacity=bus_capacity,
                                                     driver_wage=driver_wage)
    
    status, objective = solve_model(model)
    
    # Extract and analyze solution
    # ...
    
    return routes, loads, times, stats, route_plot, load_plot
end
```

For the actual optimization run, we used the following parameters:
- Maximum number of stops: 25
- Number of buses: 4
- Bus capacity: 40 passengers
- Driver wage: $25.00 per hour

The solver was set with a 5-minute time limit and a 5% optimality gap tolerance to ensure we could obtain a good feasible solution in a reasonable amount of time.



## 4. Results and discussion ##



### 4.A. Optimization Results

Our optimization model produced a feasible solution with the following key statistics:

| Metric | Value |
|--------|-------|
| Number of buses used | 4 |
| Total distance | 19,006.8 units |
| Total operation time | 651.56 minutes (~10.86 hours) |
| Average route time | 162.89 minutes (~2.7 hours) |
| Maximum route time | 294.13 minutes (~4.9 hours) |
| Maximum passenger load | 38 passengers |
| Average passenger load | 16.1 passengers |

The optimized bus routes are:

| Bus | Route | Max Load | Route Time (min) |
|-----|-------|----------|------------------|
| 1 | [988, 2894, 555, 988] | 22 | 96.31 |
| 2 | [988, 2810, 2469, 1185, 2863, 988] | 38 | 97.78 |
| 3 | [988, 877, 741, 882, 988] | 33 | 163.34 |
| 4 | [988, 628, 2866, 1757, 2896, 1262, 988] | 37 | 294.13 |

In these routes, stop 988 serves as the depot, where each bus starts and ends its journey. The buses collectively serve all 25 major stops while respecting capacity constraints. The maximum passenger load (38) is close to but does not exceed the bus capacity (40), indicating efficient utilization of resources.

It's worth noting that the optimization was terminated due to the 5-minute time limit rather than reaching the optimal solution. The final optimality gap was 69.15%, which suggests that better solutions might exist if the solver were given more time. Despite this limitation, the solution found is feasible and provides a reasonable starting point for campus shuttle route planning.



### 4.B. Sensitivity Analysis

While our main results used specific parameter values, we can discuss how changes to these parameters might affect the solution:

1. **Number of buses**: Our solution used 4 buses. Increasing this number would likely decrease the average route time and passenger load per bus, potentially improving service quality at the expense of higher operational costs. Decreasing the number of buses would force longer routes and potentially higher passenger loads, which could lead to capacity violations if reduced too much.

2. **Bus capacity**: Our model used a capacity of 40 passengers per bus. The maximum observed passenger load was 38, which is very close to this limit. Reducing the capacity would likely require redesigning routes to ensure no capacity violations, potentially increasing the number of routes needed. Increasing capacity would provide more flexibility in route design but might lead to underutilization of larger vehicles.

3. **Driver wage**: We used a wage rate of $25.00 per hour. Higher wages would increase the cost component associated with route times, potentially pushing the optimization toward solutions with shorter total operation times even at the expense of longer travel distances. Lower wages would reduce this pressure, potentially leading to longer route times if they result in shorter total distances.

4. **Time limit**: Our solution was obtained with a 5-minute solver time limit, resulting in a large optimality gap (69.15%). Increasing this limit would likely lead to better solutions with lower total costs, though with diminishing returns as the solver progresses.

The large optimality gap in our solution suggests that further improvements are possible with additional computational time or alternative solution approaches.



## 5. Conclusion ##

This project successfully developed and implemented a mixed-integer programming model for optimizing bus routes and schedules for a campus shuttle system. Our approach balanced the competing objectives of minimizing operational costs and maintaining service quality.

Key findings from our optimization include:
- A feasible solution using 4 buses to serve 25 major campus stops
- Efficient utilization of bus capacity, with maximum passenger loads near but not exceeding capacity limits
- A significant variation in route times between buses, suggesting potential for further balancing

The large optimality gap (69.15%) indicates that our solution, while feasible, is likely not the global optimum. This is a common challenge with complex MIP problems, especially given the 5-minute time limit we imposed.

For future work, we suggest several promising directions:
1. **Improve solution quality**: Allow longer computation times or explore heuristic methods to find better solutions with lower costs.
2. **Dynamic scheduling**: Extend the model to account for time-varying demand patterns throughout the day.
3. **Robust optimization**: Incorporate uncertainty in travel times and passenger demand to create more reliable schedules.
4. **Multi-objective optimization**: Explicitly model the trade-off between operational costs and service quality metrics like waiting times.
5. **Integration with real-time data**: Develop methods to adjust routes and schedules in response to real-time passenger counts and traffic conditions.

This optimization framework provides a solid foundation for campus transportation planners to develop more efficient and sustainable shuttle systems that better serve the university community while minimizing operational costs.



## 6. Author Contributions

#### 1. Modelling  
Steve Akpojisheri: 50%  
Indumathi Muruganandam: 30%  
Hunter Waugh: 20%  

#### 2. Analysis  
Steve Akpojisheri: 35%  
Indumathi Muruganandam: 35%  
Hunter Waugh: 30%  

#### 3. Data Gathering  
Steve Akpojisheri: 20%  
Indumathi Muruganandam: 30%  
Hunter Waugh: 50%  

#### 4. Software Implementation  
Steve Akpojisheri: 30%  
Indumathi Muruganandam: 35%  
Hunter Waugh: 35%  

#### 5. Report Writing    
Steve Akpojisheri: 20%  
Indumathi Muruganandam: 50%  
Hunter Waugh: 30%
